# 第 8 章: Survived データの探索

クラス分布、客室クラス・性別ごとの生存率、木の深さとクラスの重み、混同行列、分割に使われた特徴量を確かめる。

Polyglot Notebooks（.NET Interactive）は 2026 年に廃止された。この Notebook は `Microsoft.dotnet-interactive` 1.0.712001 と Plotly.NET.Interactive 5.0.0 で動作を確かめている。
先に `dotnet build` で `apps/dotnet/` のライブラリをビルドしておく。

In [ ]:
#r "nuget: FSharp.Data, 8.2.0"
#r "nuget: Plotly.NET, 5.1.0"
#r "nuget: Plotly.NET.Interactive, 5.0.0"
#r "../src/MachineLearning/bin/Debug/net10.0/MachineLearning.dll"

In [ ]:
open System.IO
open Plotly.NET
open MachineLearning.Dataset
open MachineLearning.Chapter02.IrisPreprocessing
open MachineLearning.Chapter03.DecisionTree
open MachineLearning.Chapter08.SurvivedData
open MachineLearning.Chapter08.WeightedTree
open MachineLearning.Chapter08.Pipeline

let rows = loadSurvived (Path.Combine(dataDir (), "Survived.csv"))
let x, t = splitFeaturesAndTarget rows
let split = splitTrainTest 0.2 0 x t

## クラス分布

In [ ]:
let classCounts = t |> List.countBy id |> List.sortBy fst

Chart.Column(values = (classCounts |> List.map snd), Keys = (classCounts |> List.map (fst >> string)))
|> Chart.withTitle "生存（1）と死亡（0）の人数"

## 客室クラス・性別ごとの生存率

`Survived` は 0 と 1 なので、平均値がそのまま生存率になる。

In [ ]:
let survivalRates =
    rows
    |> List.groupBy (fun row -> row.Passenger.Pclass, row.Passenger.Sex)
    |> List.map (fun ((pclass, sex), group) -> pclass, sex, group |> List.averageBy (fun row -> float row.Survived))
    |> List.sort

[ "female"; "male" ]
|> List.map (fun sex ->
    let rates = survivalRates |> List.filter (fun (_, s, _) -> s = sex)
    Chart.Column(values = (rates |> List.map (fun (_, _, rate) -> rate)), Keys = (rates |> List.map (fun (pclass, _, _) -> string pclass)), Name = sex))
|> Chart.combine
|> Chart.withTitle "客室クラス・性別ごとの生存率"
|> Chart.withXAxisStyle "Pclass"

In [ ]:
survivalRates
|> List.map (fun (pclass, sex, rate) -> {| Pclass = pclass; Sex = sex; 生存率 = System.Math.Round(rate, 3) |})
|> List.toArray

## 木の深さとクラスの重み

In [ ]:
let depths = [ 1..10 ]

let scores =
    [ Unweighted; Balanced ]
    |> List.map (fun classWeight ->
        classWeight,
        depths
        |> List.map (fun depth ->
            let options = { MaxDepth = Some depth; ClassWeight = classWeight }
            evaluate (fitPipeline options split.XTrain split.TTrain) split))

scores
|> List.map (fun (classWeight, results) ->
    Chart.Line(x = depths, y = (results |> List.map (fun result -> result.TestAccuracy)), Name = string classWeight, ShowMarkers = true))
|> Chart.combine
|> Chart.withTitle "木の深さとテストデータの正解率"
|> Chart.withXAxisStyle "深さの上限"
|> Chart.withYAxisStyle "正解率"

In [ ]:
let unweighted, balanced = snd scores[0], snd scores[1]

List.zip3 depths unweighted balanced
|> List.map (fun (depth, u, b) ->
    {| 深さ = depth
       訓練_Unweighted = System.Math.Round(u.TrainAccuracy, 3)
       訓練_Balanced = System.Math.Round(b.TrainAccuracy, 3)
       テスト_Unweighted = System.Math.Round(u.TestAccuracy, 3)
       テスト_Balanced = System.Math.Round(b.TestAccuracy, 3) |})
|> List.toArray

## 混同行列

深さ 5・`Balanced` のパイプラインで、テストデータの予測と実際を突き合わせる。

In [ ]:
let balancedPipeline =
    fitPipeline { MaxDepth = Some 5; ClassWeight = Balanced } split.XTrain split.TTrain

let pairs = List.zip (predict balancedPipeline split.XTest) split.TTest
let count pair = pairs |> List.filter ((=) pair) |> List.length

[|
    {| 実際 = "死亡"; 死亡と予測 = count (0, 0); 生存と予測 = count (1, 0) |}
    {| 実際 = "生存"; 死亡と予測 = count (0, 1); 生存と予測 = count (1, 1) |}
|]

## 分割に使われた特徴量

自作の決定木には特徴量の重要度を計算する機能が無いので、深さ 5・`Balanced` の木で、各特徴量が節の条件に使われた回数を数える。

In [ ]:
/// 木の節の条件に使われた特徴量を、根から順にすべて集める
let rec featuresOf (tree: Tree<int>) : string list =
    match tree with
    | Leaf _ -> []
    | Node(split, left, right) -> split.Feature :: featuresOf left @ featuresOf right

let featureCounts =
    featuresOf balancedPipeline.Tree |> List.countBy id |> List.sortByDescending snd

Chart.Bar(values = (featureCounts |> List.map snd), Keys = (featureCounts |> List.map fst))
|> Chart.withTitle "節の条件に使われた回数（深さ 5・Balanced）"

In [ ]:
featureCounts |> List.map (fun (feature, count) -> {| 特徴量 = feature; 回数 = count |}) |> List.toArray